In [33]:
import numpy as np, pandas as pd
from sklearn.metrics import f1_score, classification_report
import lightgbm as lgb

DATA = '.'
train = pd.read_csv(f'{DATA}/train.csv')
test = pd.read_csv(f'{DATA}/test.csv') ### Change this path for full test set
accounts = pd.read_csv(f'{DATA}/accounts.csv')
merchants = pd.read_csv(f'{DATA}/merchants.csv')
print(train.shape, test.shape)
train.head()

(1564060, 22) (260582, 21)


,txn_id,timestamp,sender_id,receiver_id,receiver_type,device_id,amount,txn_type,merchant_category,device_city,...,sender_account_age_days,new_payee,pin_retries,login_to_txn_sec,balance_before,app,app_version,ip_risk_score,qr_vpa_mismatch,label
0,TX7a868970,2026-09-04 15:37:17,AC317348,MR010440,MERCHANT,DV0317348,170.50,QR,fuel,Mathura,...,474,1,0,79,3129.28,APP_C,4.1,0.023,0.0,legit
1,TXb66b1c88,2026-08-16 16:16:20,AC228233,MR017060,MERCHANT,DV0228233,53.24,PAY,pharmacy,Ahmedabad,...,239,0,0,38,1746.10,APP_A,4.0,0.187,NaN,legit
2,TX29a953e6,2026-07-09 05:31:01,AC291565,MR016785,MERCHANT,DV0291565,153.04,QR,ecommerce,Amritsar,...,422,0,0,64,6817.73,APP_E,4.8,0.284,0.0,legit
3,TXa7fc1db5,2026-09-17 19:52:53,AC367727,MR021224,MERCHANT,DV0367727,452.94,PAY,other,Bengaluru,...,502,1,0,133,2391.89,APP_A,3.2,0.097,NaN,legit
4,TX79107a6e,2026-09-14 20:39:11,AC192491,AC364603,P2P,DV0192491,545.49,PAY,NaN,Bengaluru,...,1244,0,0,22,6443.25,APP_A,4.3,0.119,NaN,legit


In [34]:
df = pd.concat([train.assign(src='train'), test.assign(src='test')], ignore_index=True)
a, bal = df['amount'], df['balance_before']

# --- Amount features ---
df['log_amount'] = np.log1p(a)
for k in (100, 500, 1000):
    df[f'is_round_{k}'] = (a % k == 0).astype('int8')
df['has_paise'] = (a.round(2) % 1 != 0).astype('int8')
df['is_tiny'] = ((a <= 10) & (a % 1 == 0)).astype('int8')

# Placeholder limits: replace with the spikes you read off the phishing_collect histogram
LIMITS = np.array([1_000, 2_000, 5_000, 10_000, 25_000, 50_000, 100_000, np.inf])
L = LIMITS[np.searchsorted(LIMITS, a.values)]       # smallest limit >= amount
df['gap_below_limit'] = np.where(np.isinf(L), np.nan, L - a)
df['ratio_below_limit'] = np.where(np.isinf(L), np.nan, a / L)

# --- Balance features ---
df['drain_ratio'] = (a / (bal + 1)).clip(0, 2)
df['is_drain'] = (df['drain_ratio'] > 0.9).astype('int8')
df['balance_after'] = bal - a

In [35]:
# Assumed names: change the left side if yours differ
ACC_ID, ACC_CITY = 'account_id', 'home_city'
MER_ID, MER_CITY = 'merchant_id', 'city'

# Normalise city strings so "Mumbai " vs "mumbai" doesn't create fake mismatches
norm = lambda s: s.astype('string').str.strip().str.lower()
# .map keeps df's row order (a merge could reorder or duplicate rows)
df['sender_home_city'] = df['sender_id'].map(accounts.set_index(ACC_ID)[ACC_CITY])
df['merchant_city']    = df['receiver_id'].map(merchants.set_index(MER_ID)[MER_CITY]) 
dev, home, mcity = norm(df['device_city']), norm(df['sender_home_city']), norm(df['merchant_city'])

# city_ne_home: 1 if the phone is outside the sender's home city, NaN if either city is missing
df['city_ne_home'] = (dev != home).astype('float32').where(dev.notna() & home.notna())

# city_ne_merchant: NaN for P2P (no merchant city), 0/1 for merchant payments
df['city_ne_merchant'] = (dev != mcity).astype('float32').where(dev.notna() & mcity.notna())

In [36]:
def add_sender_features(df):
    sender_stats = df.groupby('sender_id').agg(
        # Amount statistics
        sender_txn_count=('amount', 'count'),
        sender_total_amount=('amount', 'sum'),
        sender_avg_amount=('amount', 'mean'),
        sender_median_amount=('amount', 'median'),
        sender_max_amount=('amount', 'max'),
        sender_min_amount=('amount', 'min'),
        sender_amount_std=('amount', 'std'),

        # Receiver statistics
        sender_unique_receivers=('receiver_id', 'nunique'),
        sender_new_payee_count=('new_payee', 'sum'),
        sender_new_payee_ratio=('new_payee', 'mean'),

        # Device, city and app statistics
        sender_unique_devices=('device_id', 'nunique'),
        sender_unique_cities=('device_city', 'nunique'),
        sender_unique_apps=('app', 'nunique'),
        sender_avg_login_to_txn_sec=('login_to_txn_sec', 'mean')
    )

    # Unique merchants
    merchant_counts = (
        df[df['receiver_type'] == 'MERCHANT']
        .groupby('sender_id')['receiver_id']
        .nunique()
        .rename('sender_unique_merchants')
    )

    # Unique P2P recipients
    p2p_counts = (
        df[df['receiver_type'] == 'P2P']
        .groupby('sender_id')['receiver_id']
        .nunique()
        .rename('sender_unique_p2p')
    )

    sender_stats = sender_stats.join(merchant_counts)
    sender_stats = sender_stats.join(p2p_counts)

    # Transaction type ratios
    txn_ratios = pd.crosstab(
        df['sender_id'],
        df['txn_type'],
        normalize='index'
    )

    txn_ratios.columns = [f'sender_{col.lower()}_ratio' for col in txn_ratios.columns]

    sender_stats = sender_stats.join(txn_ratios)

    # Merchant and P2P ratios
    receiver_ratios = pd.crosstab(
        df['sender_id'],
        df['receiver_type'],
        normalize='index'
    )

    receiver_ratios.columns = [
        f'sender_{col.lower()}_ratio' for col in receiver_ratios.columns
    ]

    sender_stats = sender_stats.join(receiver_ratios)

    # Join features to original dataframe
    df = df.join(sender_stats, on='sender_id')

    # Handle missing values
    df = df.fillna({
        'sender_amount_std': 0,
        'sender_unique_merchants': 0,
        'sender_unique_p2p': 0
    })

    return df

In [37]:
def add_receiver_features(df):

    # A. Incoming transaction statistics
    incoming_stats = df.groupby('receiver_id').agg(
        receiver_txn_count=('amount', 'count'),
        receiver_unique_senders=('sender_id', 'nunique'),
        receiver_total_received=('amount', 'sum'),
        receiver_avg_received=('amount', 'mean'),
        receiver_median_received=('amount', 'median'),
        receiver_max_received=('amount', 'max'),
        receiver_amount_std=('amount', 'std')
    )

    # B. Outgoing transaction statistics
    outgoing_stats = df.groupby('sender_id').agg(
        receiver_outgoing_count=('amount', 'count'),
        receiver_outgoing_amount=('amount', 'sum'),
        receiver_unique_outgoing=('receiver_id', 'nunique'),
        receiver_avg_outgoing=('amount', 'mean')
    )

    # Merge incoming and outgoing statistics
    receiver_stats = incoming_stats.join(outgoing_stats)

    # C. Incoming vs outgoing behaviour

    # Count ratio
    receiver_stats['receiver_in_out_count_ratio'] = (
        receiver_stats['receiver_txn_count'] /
        receiver_stats['receiver_outgoing_count'].replace(0, np.nan)
    )

    # Amount ratio
    receiver_stats['receiver_in_out_amount_ratio'] = (
        receiver_stats['receiver_total_received'] /
        receiver_stats['receiver_outgoing_amount'].replace(0, np.nan)
    )

    # Net amount
    receiver_stats['receiver_net_amount'] = (
        receiver_stats['receiver_total_received'] -
        receiver_stats['receiver_outgoing_amount'].fillna(0)
    )

    # Outgoing amount relative to incoming amount
    receiver_stats['receiver_outgoing_share'] = (
        receiver_stats['receiver_outgoing_amount'] /
        receiver_stats['receiver_total_received'].replace(0, np.nan)
    )

    # Join receiver statistics to original dataframe
    df = df.join(receiver_stats, on='receiver_id')

    # Handle missing values
    fill_cols = [
        'receiver_outgoing_count',
        'receiver_outgoing_amount',
        'receiver_unique_outgoing',
        'receiver_avg_outgoing',
        'receiver_amount_std',
        'receiver_in_out_count_ratio',
        'receiver_in_out_amount_ratio',
        'receiver_outgoing_share'
    ]

    df[fill_cols] = df[fill_cols].fillna(0)

    return df

In [38]:
def add_device_features(df):

    # A. Device activity statistics
    device_stats = df.groupby('device_id').agg(
        device_txn_count=('amount', 'count'),
        device_total_amount=('amount', 'sum'),
        device_avg_amount=('amount', 'mean'),
        device_amount_std=('amount', 'std'),

        device_unique_senders=('sender_id', 'nunique'),
        device_unique_receivers=('receiver_id', 'nunique'),
        device_unique_cities=('device_city', 'nunique'),
        device_unique_apps=('app', 'nunique'),
        device_avg_login_to_txn_sec=('login_to_txn_sec', 'mean')
    )

    # B. Device-account relationships

    # Number of distinct devices associated with each sender
    sender_devices = (
        df.groupby('sender_id')['device_id']
        .nunique()
        .rename('sender_devices')
    )

    # Number of transactions made by each sender on each device
    sender_device_txns = (
        df.groupby(['device_id', 'sender_id'])
        .size()
        .rename('sender_device_txn_count')
        .reset_index()
    )

    # Combine sender-device transaction counts with device totals
    sender_device_txns = sender_device_txns.join(
        device_stats['device_txn_count'],
        on='device_id'
    )

    # Proportion of device transactions made by the current sender
    sender_device_txns['device_sender_txn_share'] = (
        sender_device_txns['sender_device_txn_count'] /
        sender_device_txns['device_txn_count']
    )

    # Join device-level statistics to sender-device combinations
    sender_device_features = sender_device_txns.set_index(
        ['device_id', 'sender_id']
    )

    # Map device statistics to transactions
    df = df.join(device_stats, on='device_id')

    # Map sender-device relationship features to transactions
    df = df.join(
        sender_device_features[
            ['device_sender_txn_share']
        ],
        on=['device_id', 'sender_id']
    )

    # Map sender's unique device count
    df = df.join(sender_devices, on='sender_id')

    # C. Multi-account device indicator
    df['device_multi_account'] = (
        df['device_unique_senders'] > 1
    ).astype(int)

    # Handle missing standard deviations
    df['device_amount_std'] = df['device_amount_std'].fillna(0)

    return df

In [39]:
def add_merchant_features(df):

    # Only consider merchant transactions
    merchant_df = df[df['receiver_type'] == 'MERCHANT'].copy()

    # A. Merchant transaction statistics
    merchant_stats = merchant_df.groupby('receiver_id').agg(
        merchant_txn_count=('amount', 'count'),
        merchant_unique_customers=('sender_id', 'nunique'),
        merchant_total_amount=('amount', 'sum'),
        merchant_avg_amount=('amount', 'mean'),
        merchant_median_amount=('amount', 'median'),
        merchant_amount_std=('amount', 'std'),
        merchant_max_amount=('amount', 'max'),

        # Geographic and device diversity
        merchant_unique_customer_cities=('device_city', 'nunique'),
        merchant_unique_devices=('device_id', 'nunique')
    )

    # B. Merchant behaviour

    # QR transaction ratio
    merchant_qr_ratio = (
        merchant_df.assign(is_qr=merchant_df['txn_type'] == 'QR')
        .groupby('receiver_id')['is_qr']
        .mean()
        .rename('merchant_qr_ratio')
    )

    # Number of refunds
    merchant_refund_count = (
        merchant_df[merchant_df['txn_type'] == 'REFUND']
        .groupby('receiver_id')
        .size()
        .rename('merchant_refund_count')
    )

    # Proportion of round-number transactions (multiples of ₹100)
    merchant_round_amount_ratio = (
        merchant_df.assign(
            is_round=merchant_df['amount'] % 100 == 0
        )
        .groupby('receiver_id')['is_round']
        .mean()
        .rename('merchant_round_amount_ratio')
    )

    # Combine all merchant features
    merchant_stats = merchant_stats.join(merchant_qr_ratio)
    merchant_stats = merchant_stats.join(merchant_refund_count)
    merchant_stats = merchant_stats.join(merchant_round_amount_ratio)

    # Join merchant statistics to original dataframe
    df = df.join(merchant_stats, on='receiver_id')

    # Handle missing values for non-merchants
    fill_cols = [
        'merchant_txn_count',
        'merchant_unique_customers',
        'merchant_total_amount',
        'merchant_avg_amount',
        'merchant_median_amount',
        'merchant_amount_std',
        'merchant_max_amount',
        'merchant_qr_ratio',
        'merchant_refund_count',
        'merchant_round_amount_ratio',
        'merchant_unique_customer_cities',
        'merchant_unique_devices'
    ]

    df[fill_cols] = df[fill_cols].fillna(0)

    return df

In [40]:
def device_rolling_txn(df, window="5min", device_col="device_id",
                       timestamp_col="timestamp", amount_col="amount"):
    
    out = df.copy()
    ts = pd.to_datetime(out[timestamp_col])
    amt = out[amount_col].to_numpy(dtype="float64")

    # integer codes per device (NaN gets its own code instead of being dropped)
    codes = pd.factorize(out[device_col], use_na_sentinel=False)[0].astype("int64")

    # whole seconds since the earliest timestamp
    t = ((ts - ts.min()) // pd.Timedelta("1s")).to_numpy(dtype="int64")
    W = int(pd.Timedelta(window).total_seconds())

    # One global sorted key: device code * span + time.
    # span is wide enough that a window never reaches into the previous device's block.
    span = int(t.max()) + W + 1
    key = codes * span + t

    order = np.argsort(key, kind="stable")      # ties keep original row order
    k, a = key[order], amt[order]

    # window (t - W, t]: lo excludes exactly t-W, hi includes all rows with time == t
    lo = np.searchsorted(k, k - W, side="right")
    hi = np.searchsorted(k, k, side="right")

    cs = np.concatenate(([0.0], np.cumsum(a)))  # prefix sums
    prev_cnt = hi - lo - 1                      # remove the current row
    prev_sum = cs[hi] - cs[lo] - a

    avg = np.full(len(k), np.nan)
    m = prev_cnt > 0
    avg[m] = prev_sum[m] / prev_cnt[m]

    # scatter back to the original row order
    cnt_out = np.empty(len(k), dtype="int32")
    avg_out = np.empty(len(k), dtype="float64")
    cnt_out[order] = prev_cnt
    avg_out[order] = avg

    prefix = device_col.replace("_id", "")
    out[f"{prefix}_txn_count_{window}"] = cnt_out
    out[f"{prefix}_avg_amt_{window}"] = avg_out
    return out

In [41]:
for col in ["sender_id", "device_id","receiver_id"]:
    for win in ["5min", "10min", "1h", "24h"]:
        df = device_rolling_txn(df, window=win, device_col = col)

In [42]:
def interarrival_features(df, entity_col, prefix):
    out = df.copy()
    n = len(out)

    sec = pd.to_datetime(out['timestamp']).to_numpy('datetime64[s]').astype('int64')
    code = pd.factorize(out[entity_col])[0].astype('int64')

    o = np.lexsort((np.arange(n), sec, code))          # sort by entity, time, original row
    c, t = code[o], sec[o]

    same = np.r_[False, c[1:] == c[:-1]]
    gap = np.where(same, t - np.r_[0, t[:-1]], np.nan)  # seconds since this entity's previous txn

    # The 5 gaps BEFORE the current row (your shift(1).rolling(5)): lag k = gap of the row k back
    G = np.full((n, 5), np.nan)
    for k in range(1, 6):
        ok = np.r_[np.zeros(k, bool), c[k:] == c[:n - k]]
        G[:, k - 1] = np.where(ok, np.r_[np.full(k, np.nan), gap[:n - k]], np.nan)

    with np.errstate(all='ignore'):
        valid = ~np.isnan(G); cnt = valid.sum(1)
        mean = np.nansum(G, 1) / cnt                                       # min_periods=1
        mn   = np.where(cnt >= 1, np.where(valid, G, np.inf).min(1), np.nan)
        sd   = np.where(cnt >= 2,                                          # min_periods=2, ddof=1
                        np.sqrt(np.nansum((G - mean[:, None]) ** 2, 1) / (cnt - 1)), np.nan)

    def unsort(a):
        r = np.empty(n, 'float32'); r[o] = a; return r

    new = pd.DataFrame({
        f'{prefix}_time_since_last_txn': unsort(gap),
        f'{prefix}_avg_interarrival_5':  unsort(mean),
        f'{prefix}_min_interarrival_5':  unsort(mn),
        f'{prefix}_std_interarrival_5':  unsort(sd),
    }, index=out.index)

    out = out.drop(columns=new.columns.intersection(out.columns))   # safe to re-run
    return pd.concat([out, new], axis=1)

In [43]:
def refund_count_30d(d, window='30D'):
    ts = pd.to_datetime(d.timestamp)
    w = int(pd.Timedelta(window) / pd.Timedelta('1ms'))
    ms = ((ts - ts.min()) // pd.Timedelta('1ms')).to_numpy('int64')
    codes = pd.factorize(d.sender_id)[0].astype('int64')

    # one sorted key per row: sender block first, then time within the block.
    # the +w shift keeps (t - window) from spilling into the previous sender's block
    span = ms.max() + 2 * w + 1
    key = codes * span + ms + w

    order = np.argsort(key, kind='stable')
    ks = key[order]
    is_ref = (d.txn_type.to_numpy() == 'REFUND').astype('int64')[order]
    cum = np.concatenate([[0], np.cumsum(is_ref)])

    hi = np.searchsorted(ks, ks, side='left')       # rows strictly before t
    lo = np.searchsorted(ks, ks - w, side='left')   # rows at or after t - window

    out = np.empty(len(d), dtype='int64')
    out[order] = cum[hi] - cum[lo]
    return pd.Series(out, index=d.index)

df['sender_refund_count_30d'] = refund_count_30d(df).to_numpy()

In [44]:
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Only account-to-account transfers
p2p = df[
    (df['receiver_type'] == 'P2P') &
    df['sender_id'].notna() &
    df['receiver_id'].notna()
].copy()

In [45]:
# Incoming transactions
incoming = p2p.groupby('receiver_id').agg(
    chain_in_txn_count=('txn_id', 'count'),
    chain_in_amount=('amount', 'sum'),
    chain_in_unique_accounts=('sender_id', 'nunique'),
    chain_in_avg_amount=('amount', 'mean'),
    chain_in_max_amount=('amount', 'max')
)

# Outgoing transactions
outgoing = p2p.groupby('sender_id').agg(
    chain_out_txn_count=('txn_id', 'count'),
    chain_out_amount=('amount', 'sum'),
    chain_out_unique_accounts=('receiver_id', 'nunique'),
    chain_out_avg_amount=('amount', 'mean'),
    chain_out_max_amount=('amount', 'max')
)

incoming.index.name = 'account_id'
outgoing.index.name = 'account_id'

In [46]:
def add_chain_features(df):
    X = pd.DataFrame(index=df.index)

    # Upstream: activity flowing into the sender
    for col in incoming.columns:
        X[f'sender_{col}'] = df['sender_id'].map(incoming[col])

    # Downstream: activity flowing out of the receiver
    for col in outgoing.columns:
        X[f'receiver_{col}'] = df['receiver_id'].map(outgoing[col])

    # Replace missing account history with zero
    X = X.fillna(0)

    # How many accounts are upstream and downstream?
    X['chain_upstream_degree'] = X['sender_chain_in_unique_accounts']
    X['chain_downstream_degree'] = X['receiver_chain_out_unique_accounts']

    # Number of possible A -> B -> C paths
    X['chain_2hop_paths'] = X['chain_downstream_degree']

    # Number of possible X -> A -> B -> C paths
    X['chain_3hop_paths'] = (
        X['chain_upstream_degree'] *
        X['chain_downstream_degree']
    )

    # Amount relationships
    amount = df['amount'].replace(0, np.nan)

    X['chain_sender_in_amount_ratio'] = (
        X['sender_chain_in_amount'] / amount
    )

    X['chain_receiver_out_amount_ratio'] = (
        X['receiver_chain_out_amount'] / amount
    )

    X['chain_sender_in_avg_ratio'] = (
        X['sender_chain_in_avg_amount'] / amount
    )

    X['chain_receiver_out_avg_ratio'] = (
        X['receiver_chain_out_avg_amount'] / amount
    )

    X['chain_net_flow_around_txn'] = (
        X['receiver_chain_out_amount'] -
        X['sender_chain_in_amount']
    )

    return X.replace([np.inf, -np.inf], np.nan).fillna(0)

In [47]:
df = add_sender_features(df)

In [48]:
df = add_receiver_features(df)

In [49]:
df = add_device_features(df)

In [50]:
df = add_merchant_features(df)

In [51]:
df = interarrival_features(df, 'sender_id',   'sender')
df = interarrival_features(df, 'device_id',   'device')
df = interarrival_features(df, 'receiver_id', 'receiver')

In [52]:
chain_features = add_chain_features(df)

df = pd.concat([df, chain_features], axis=1)

In [53]:
train_fe = df[df['src'] == 'train'].drop(columns='src').reset_index(drop=True)
test_fe = df[df['src'] == 'test'].drop(columns='src').reset_index(drop=True)

# Test has no labels, so drop the all-NaN label column that concat created
test_fe = test_fe.drop(columns='label')

In [54]:
home = accounts.set_index('account_id').home_city
CATS = ['receiver_type', 'txn_type', 'merchant_category', 'app']
exclude = CATS + ['txn_id', 'timestamp', 'sender_id', 'receiver_id',
                  'device_id', 'device_city', 'sender_home_city', 'merchant_city', 'label']

def row_features(d):
    X = pd.DataFrame(index=d.index)
    t = pd.to_datetime(d.timestamp)
    X['hour'] = t.dt.hour
    X['weekday'] = t.dt.day_of_week
    for c in d.columns:
        if c not in exclude:
        # if c in f95:
            X[c] = d[c]
    # X['amount_share_of_balance'] = d.amount / d.balance_before
    # X['away_from_home'] = (d.device_city != d.sender_id.map(home)).astype(int)
    for c in CATS:
        X[c] = pd.Categorical(d[c].fillna('none'), categories=sorted(set(train[c].fillna('none')) | set(test[c].fillna('none')))).codes
    return X

X, Xt = row_features(train_fe), row_features(test_fe)
X.head()

C:\Users\aryan\AppData\Local\Temp\ipykernel_35632\3402058201.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X[c] = d[c]
C:\Users\aryan\AppData\Local\Temp\ipykernel_35632\3402058201.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X[c] = d[c]
C:\Users\aryan\AppData\Local\Temp\ipykernel_35632\3402058201.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instea

,hour,weekday,amount,sim_age_hours,device_age_days,sender_account_age_days,new_payee,pin_retries,login_to_txn_sec,balance_before,...,chain_3hop_paths,chain_sender_in_amount_ratio,chain_receiver_out_amount_ratio,chain_sender_in_avg_ratio,chain_receiver_out_avg_ratio,chain_net_flow_around_txn,receiver_type,txn_type,merchant_category,app
0,15,4,170.50,15549.4,1.2,474,1,0,79,3129.28,...,0.0,100.762463,0.0,50.381232,0.0,-17180.00,0,3,4,2
1,16,6,53.24,13222.7,228.2,239,0,0,38,1746.10,...,0.0,5.822690,0.0,5.822690,0.0,-310.00,0,2,10,0
2,5,3,153.04,2490.7,27.2,422,0,0,64,6817.73,...,0.0,0.000000,0.0,0.000000,0.0,0.00,0,3,0,4
3,19,3,452.94,5745.0,394.7,502,1,0,133,2391.89,...,0.0,0.303087,0.0,0.303087,0.0,-137.28,0,2,9,0
4,20,0,545.49,320.0,33.1,1244,0,0,22,6443.25,...,0.0,2.236521,0.0,2.236521,0.0,-1220.00,1,2,8,0


In [ ]:
from sklearn.metrics import f1_score, classification_report
from sklearn.ensemble import HistGradientBoostingClassifier

import lightgbm as lgb

is_sep = train.timestamp.str[:7] == '2026-09'

model = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.1,
    num_leaves=31,          # roughly HGB's max_leaf_nodes=31
    min_child_samples=20,
    subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=0,
    n_jobs=-1,
    verbose=-1,
)
model.fit(X[~is_sep], train.label[~is_sep])

# --- probabilities instead of predict() ---
proba = model.predict_proba(X[is_sep])
classes = model.classes_
y_val = train.label[is_sep]
y_int = pd.Categorical(y_val, categories=classes).codes      # ints, much faster than strings
legit_idx = list(classes).index('legit')

print('baseline macro-F1:', round(f1_score(y_int, proba.argmax(1), average='macro'), 4))

# --- split September in half: tune on one, score on the other ---
rng = np.random.RandomState(0)
half = rng.rand(len(y_int)) < 0.5
# (if rows are time-ordered, a cleaner split is by date: first half of Sep vs second half)

def score(w, mask):
    return f1_score(y_int[mask], (proba[mask] * w).argmax(1), average='macro')

w = np.ones(len(classes))
best = score(w, half)
grid = [0.5, 0.7, 1, 1.5, 2, 3, 5, 8, 12, 20]
for _ in range(3):
    for i in range(len(classes)):
        if i == legit_idx:
            continue
        for g in grid:
            w2 = w.copy(); w2[i] = g
            s = score(w2, half)
            if s > best:
                best, w = s, w2

print('tuned on half A :', round(best, 4))
print('held-out half B :', round(score(w, ~half), 4), '(baseline B:', round(score(np.ones(len(classes)), ~half), 4), ')')
print(dict(zip(classes, w.round(2))))

# --- final report using the weights ---
pred = classes[(proba * w).argmax(1)]
print(classification_report(y_val, pred, digits=3, zero_division=0))

In [ ]:
imp = pd.DataFrame({
    'feature': X.columns,
    'gain': model.booster_.feature_importance('gain'),
    'splits': model.booster_.feature_importance('split'),
}).sort_values('gain', ascending=False)

imp['gain_pct'] = imp.gain / imp.gain.sum() * 100
imp['cum_pct'] = imp.gain_pct.cumsum()
print(imp.head(40).round(2).to_string(index=False))
print('Features with 0 gain:', (imp.gain == 0).sum())
print('Features needed for 95% of gain:', (imp.cum_pct < 95).sum() + 1)
print('Features needed for 99% of gain:', (imp.cum_pct < 99).sum() + 1)

features_95 = imp.loc[imp.cum_pct <= 95, 'feature'].tolist()
features_99 = imp.loc[imp.cum_pct <= 99, 'feature'].tolist()

print('\n--- Features covering 95% gain ---')
print(features_95)

print('\n--- Features covering 99% gain ---')
print(features_99)

                        feature       gain  splits  gain_pct  cum_pct
            device_txn_count_1h 1921023.02      78     17.77    17.77
     device_time_since_last_txn 1327633.08    1228     12.28    30.06
      device_avg_interarrival_5  912984.46     762      8.45    38.51
           device_txn_count_24h  821809.64      78      7.60    46.11
          device_txn_count_5min  499498.76     115      4.62    50.73
                gap_below_limit  497007.96    1325      4.60    55.33
         device_txn_count_10min  487085.92      90      4.51    59.84
                sender_qr_ratio  411721.36     946      3.81    63.65
             device_avg_amt_24h  277898.64     311      2.57    66.22
                qr_vpa_mismatch  197217.12     214      1.82    68.04
                  balance_after  193297.38    1140      1.79    69.83
              ratio_below_limit  182084.73    1155      1.68    71.52
            receiver_avg_amt_1h  152792.68     796      1.41    72.93
                devi

In [ ]:
model.fit(X, train.label)
sub = pd.DataFrame({'id': test.txn_id, 'label': model.predict(Xt)})
sub.to_csv('submission.csv', index=False)
print(sub.label.value_counts())
print('wrote submission.csv', sub.shape)

KeyboardInterrupt: 